# AudioCraft / AudioGen — safe run guide

Use the `phase1` Jupyter kernel and run the notebook **top to bottom**. The environment audit on October 11, 2026 found AudioCraft 1.3.0 with PyTorch / torchvision / torchaudio 2.1.0 + CUDA 11.8, and the AudioGen imports passed. Do not run a blanket uninstall/reinstall or `pip install -U audiocraft` against this shared environment.

The setup cells below only inspect the active kernel. They do not install, uninstall, or upgrade packages. If a dependency check fails, save the complete error and repair a cloned/isolated environment rather than modifying the shared `phase1` environment blindly. The recorded `pip check` conflict is `thinc 8.3.6` requiring NumPy >=2 while this AudioCraft stack pins NumPy 1.26.4; that conflict has **not** been shown to cause the earlier wheel-build report, and AudioCraft imports currently work.

AudioGen medium is about 3.91 GB on the Hugging Face model repository and AudioCraft documents at least 16 GB of GPU memory for medium models. The model cell checks the local Hub cache first; a missing snapshot requires OEC StorageManager capacity admission before downloading. The AudioGen model weights are CC-BY-NC 4.0; check the model licence and suitability before commercial use. See the [model card](https://huggingface.co/facebook/audiogen-medium) and [AudioCraft AudioGen documentation](https://github.com/facebookresearch/audiocraft/blob/main/docs/AUDIOGEN.md).

The Gradio UI uses port 7861 (public share off). On a shared compute node, use an SSH tunnel from your local machine after substituting the actual compute host:
```bash
ssh -N -L 7861:<COMPUTE_NODE>:7861 rahkus@login.npsf.cdac.in
```


In [ ]:
import os
import sys
from pathlib import Path

import core_config
from core_config import SYSTEM_DIRS

# Explicitly use OEC's resolved output/cache paths.
AUDIO_OUTPUT_DIR = Path(SYSTEM_DIRS["outputs"]) / "audio"
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PYTHON_EXECUTABLE =", sys.executable)
print("AUDIOCRAFT_CACHE_DIR =", os.environ.get("AUDIOCRAFT_CACHE_DIR"))
print("MODEL_CACHE_ROOT =", SYSTEM_DIRS["models"])
print("AUDIO_OUTPUT_DIR =", AUDIO_OUTPUT_DIR)
print("[SYSTEM] Core paths and proxies are configured for this kernel.")


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

expected = {
    "audiocraft": "1.3.0",
    "torch": "2.1.0",
    "torchvision": "0.16.0",
    "torchaudio": "2.1.0",
    "numpy": "1.26.4",
    "transformers": "4.39.3",
}
actual = {}
for package, required_prefix in expected.items():
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"Required package {package!r} is missing from {sys.executable}. "
            "This notebook does not auto-install into the shared environment."
        ) from exc
    actual[package] = version
    print(f"{package}=={version}")
    if not version.startswith(required_prefix):
        raise RuntimeError(
            f"{package} version {version} does not match expected prefix {required_prefix}. "
            "Select the correct phase1 kernel; do not bulk-upgrade this environment."
        )

import torch
assert torch.version.cuda and torch.version.cuda.startswith("11.8"), (
    f"Expected phase1 PyTorch CUDA 11.8 build, got torch={torch.__version__} cuda={torch.version.cuda}"
)
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write
print("AUDIOCRAFT_IMPORT_CHECK=PASS")
print("TORCH_CUDA_AVAILABLE =", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("AudioGen medium inference requires a usable GPU in the selected phase1 environment.")

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
if pip_check.returncode:
    print("PIP_CHECK_WARNING_BEGIN")
    print(pip_check.stdout.strip())
    print("PIP_CHECK_WARNING_END")
    print("Dependency metadata conflicts were reported; no package was changed.")
else:
    print("PIP_CHECK=PASS")


In [ ]:
import importlib.metadata as metadata
import gradio as gr

gradio_version = metadata.version("gradio")
print("GRADIO_VERSION =", gradio_version)
print("GRADIO_IMPORT_CHECK=PASS")


In [ ]:
import gc
import os
import uuid
from pathlib import Path

import gradio as gr
import torch
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write
from huggingface_hub import snapshot_download

try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

from storage_manager import managed_storage_operation, storage_preflight
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

AUDIOGEN_MODEL_ID = "facebook/audiogen-medium"
AUDIOGEN_CACHE_ROOT = Path(os.environ.get("AUDIOCRAFT_CACHE_DIR", SYSTEM_DIRS["models"])).expanduser().resolve()
AUDIOGEN_OUTPUT_DIR = Path(SYSTEM_DIRS["outputs"]).expanduser().resolve() / "audio"


def ensure_audiogen_snapshot():
    """Return a local Hub snapshot; admit any model download before bytes are fetched."""
    try:
        snapshot = snapshot_download(
            repo_id=AUDIOGEN_MODEL_ID,
            cache_dir=str(AUDIOGEN_CACHE_ROOT),
            local_files_only=True,
        )
        print(f"AUDIOGEN_CACHE=HIT snapshot={snapshot}")
        return snapshot
    except LocalEntryNotFoundError:
        pass

    # The model repository is ~3.91 GB; add configurable headroom for metadata,
    # temporary files, and the AudioCraft cache rather than assuming free storage.
    safety_gb = max(0.0, float(os.environ.get("AIEROS_AUDIOGEN_DOWNLOAD_SAFETY_GB", "1.5")))
    required_gb = estimate_hf_repo_bytes(AUDIOGEN_MODEL_ID) / BYTES_PER_GB + safety_gb
    minimum_free_gb = max(0.5, float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")))
    ok, reason = storage_preflight(
        "models",
        required_gb=required_gb,
        require_fresh_quota=True,
        minimum_free_gb=minimum_free_gb,
    )
    if not ok:
        raise RuntimeError(
            "AudioGen model is not cached, and OEC denied the download because storage admission "
            f"did not pass: {reason}. A fresh authoritative quota snapshot is required; filesystem free space alone is not sufficient. Do not bypass this guard."
        )
    ok, reason = storage_preflight(
        "models",
        required_gb=required_gb,
        require_fresh_quota=True,
        minimum_free_gb=minimum_free_gb,
    )
    if not ok:
            raise RuntimeError(f"Storage admission denied before managed write: {reason}")
    with managed_storage_operation(
        "models",
        required_gb=required_gb,
        require_fresh_quota=True,
        minimum_free_gb=minimum_free_gb,
    ):
        ok, reason = storage_preflight(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=minimum_free_gb,
        )
        if not ok:
            raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
        snapshot = snapshot_download(
            repo_id=AUDIOGEN_MODEL_ID,
            cache_dir=str(AUDIOGEN_CACHE_ROOT),
        )
    print(f"AUDIOGEN_DOWNLOAD=PASS snapshot={snapshot}")
    return snapshot


print("Checking AudioGen model cache / storage admission...")
AUDIOGEN_SNAPSHOT = ensure_audiogen_snapshot()
print("Loading AudioGen medium. This needs a GPU with roughly 16 GB or more available VRAM.")
model = AudioGen.get_pretrained(AUDIOGEN_MODEL_ID, device="cuda")
model.set_generation_params(duration=5)
print("AUDIOGEN_MODEL_LOAD=PASS")


def generate_sfx(prompt: str, duration: int):
    if not prompt or not prompt.strip():
        raise gr.Error("Enter a sound-effect description first.")
    if not 1 <= int(duration) <= 10:
        raise gr.Error("Duration must be between 1 and 10 seconds.")
    try:
        model.set_generation_params(duration=int(duration))
        with torch.inference_mode():
            wav = model.generate([prompt.strip()])
        output_path = AUDIOGEN_OUTPUT_DIR / f"audiogen_{uuid.uuid4().hex[:12]}.wav"
        ok, reason = storage_preflight(
            "outputs",
            required_gb=0.03,
            require_fresh_quota=True,
            minimum_free_gb=0.02,
        )
        if not ok:
                raise RuntimeError(f"Storage admission denied before managed write: {reason}")
        with managed_storage_operation(
            "outputs",
            required_gb=0.03,
            require_fresh_quota=True,
            minimum_free_gb=0.02,
        ):
            ok, reason = storage_preflight(
                "outputs",
                required_gb=0.03,
                require_fresh_quota=True,
                minimum_free_gb=0.02,
            )
            if not ok:
                raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
            AUDIOGEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            audio_write(
                str(output_path.with_suffix("")),
                wav[0].detach().float().cpu(),
                model.sample_rate,
                strategy="loudness",
            )
        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(f"AudioCraft did not create a non-empty WAV file: {output_path}")
        print(f"AUDIOGEN_OUTPUT={output_path} bytes={output_path.stat().st_size}")
        gc.collect()
        torch.cuda.empty_cache()
        return str(output_path)
    except gr.Error:
        raise
    except Exception as exc:
        raise gr.Error(f"AudioGen generation failed: {type(exc).__name__}: {exc}") from exc


with gr.Blocks(title="OEC AudioCraft AudioGen", theme=gr.themes.Monochrome()) as demo:
    gr.Markdown(
        "# OEC AudioCraft / AudioGen\n"
        "Generate one short sound effect at a time. The Gradio share URL is disabled by default; "
        "use the SSH tunnel from the first cell on shared compute."
    )
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(
                label="Sound-effect description",
                value="cinematic deep bass impact with metallic sci-fi resonance",
                lines=3,
            )
            duration = gr.Slider(
                label="Duration (seconds)", minimum=1, maximum=10, value=5, step=1
            )
            generate_btn = gr.Button("Generate audio", variant="primary")
        with gr.Column():
            output_audio = gr.Audio(label="Generated WAV", autoplay=False)
            output_file = gr.File(label="Download WAV")

    def generate_sfx_outputs(prompt_text: str, duration_seconds: int):
        path = generate_sfx(prompt_text, int(duration_seconds))
        return path, path

    generate_btn.click(
        fn=generate_sfx_outputs,
        inputs=[prompt, duration],
        outputs=[output_audio, output_file],
    )

print("Starting AudioGen Gradio server on 0.0.0.0:7861 (share=False).")
demo.launch(server_name="0.0.0.0", server_port=7861, inline=False, share=False, show_error=True)


In [ ]:
# Intentionally no shutdown call here: the Gradio server is started by the previous cell.
# Stop it with the Jupyter interrupt button when you are finished.
